In [1]:
import pandas as pd
import numpy as np
import Functions as fn

In [ ]:
df_stories = pd.read_parquet('Path')

In [3]:
A, P, A_id, A_name, id_to_name = fn.adjacency_matrix(df_stories, team_id=6803)

idx_to_player_id = A_id.index.to_list()                
player_id_to_idx = {pid: i for i, pid in enumerate(idx_to_player_id)}  

idx_to_player_name = [id_to_name.get(pid, str(pid)) for pid in idx_to_player_id]

S, S_df = fn.signed_balance_matrix(df_stories, idx_to_player_id, id_to_name)
S_df.head()

D = fn.most_probable_path_matrix(P)

In [4]:
'''
Groups of vertices
'''

'\nGroups of vertices\n'

In [ ]:
keep, core_nodes = fn.k_core(A, k_in=16, k_out=16)
kcore_df = pd.DataFrame({
    'player_id': np.array(idx_to_player_id)[keep],
    'player_name': [id_to_name.get(pid, None) for pid in np.array(idx_to_player_id)[keep]]
})
kcore_df.head(30)

,player_id,player_name
0,52072,Jonna Ann-Charlotte Andersson
1,302840,Asato Miyagawa
2,348018,Vilde Hasund
3,351320,Eva Nyström
4,371014,Emma Westin
5,401725,Alice Carlsson
6,418723,Stina Lennartsson
7,520772,Anna Langås Jøsendal
8,535334,Julie Blakstad
9,566534,Emilie Marie Joramo


In [ ]:
kcore_df.to_csv('kcore_players.csv', index=False)

In [7]:
'''
Reciprocity
'''

'\nReciprocity\n'

In [8]:
r = fn.reciprocity(P)
print(r)

R_pair = fn.reciprocity_pairwise(P, eps=1e-12)
R_pair = np.asarray(R_pair, dtype=float)

player_names = [id_to_name.get(pid, None) for pid in idx_to_player_id]

R_pair_df = pd.DataFrame(R_pair, index=player_names, columns=player_names)
R_pair_df.head()

0.8704663212435233


,None,None,Jonna Ann-Charlotte Andersson,Ellen Gibson,Julia Elisabeth Roddar,Nadia Nadim,Simone Boye-Sørensen,Lotta Ökvist,Asato Miyagawa,Vilde Hasund,...,Bella Andersson,Bea Sprung,Suzu Amano,Smilla Holmberg,Lykke Ihrfelt,Vera Elin Kristina Blom,Doris Petz,Sally Nylén,Stella Maiquez,Fanny Peterson
None,0.000000,0.000000,23.659831,0.000000,22.908068,0.0,22.241949,24.197034,23.608001,23.998381,...,22.183207,23.257363,0.000000,23.324574,0.000000,0.0,0.0,0.0,0.0,0.0
None,0.000000,0.000000,25.066072,0.000000,0.000000,0.0,25.066072,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
Jonna Ann-Charlotte Andersson,23.659831,25.066072,0.000000,2.515903,0.793136,0.0,0.000000,0.000000,0.470303,0.505123,...,0.360295,0.000000,4.182499,1.222560,0.000000,0.0,0.0,0.0,0.0,0.0
Ellen Gibson,0.000000,0.000000,2.515903,0.000000,1.801810,0.0,0.000000,0.000000,1.647103,2.503256,...,2.826776,0.000000,24.797808,2.745211,26.714730,0.0,0.0,0.0,0.0,0.0
Julia Elisabeth Roddar,22.908068,0.000000,0.793136,1.801810,0.000000,0.0,0.000000,1.982113,0.000000,0.737814,...,1.012543,0.000000,0.000000,1.260936,21.521774,0.0,0.0,0.0,0.0,0.0


In [ ]:
R_pair_df.to_csv('reciprocity_matrix.csv', index=False)

In [10]:
'''
Structural balance value
'''

'\nStructural balance value\n'

In [11]:
b = fn.structural_balance_global(S)
print(b)

0.7064220183486238


In [12]:
'''
Which players are in the groups?
'''

'\nWhich players are in the groups?\n'

In [ ]:
ok, group, clusters = fn.structural_balance_groups(S)

print('Balanced:', ok)
print('Groups:', group)
print('Clusters:', clusters)

if ok:
    balance_df = pd.DataFrame({
        'player_id': idx_to_player_id,
        'player_name': [id_to_name.get(pid, None) for pid in idx_to_player_id],
        'balance_group': group
    }).sort_values(['balance_group', 'player_name'])

    balance_df.head(20)

Balanced: False
Groups: None
Clusters: None


In [14]:
'''
How similar are the players in terms of their connections?
'''

'\nHow similar are the players in terms of their connections?\n'

In [ ]:
pearson = fn.pearson_matrix(P, mode='out')

pearson = np.asarray(pearson, dtype=float)

player_names = [id_to_name.get(pid, None) for pid in idx_to_player_id]

pearson_df = pd.DataFrame(pearson, index=player_names, columns=player_names)

pearson_df.head()

,None,None,Jonna Ann-Charlotte Andersson,Ellen Gibson,Julia Elisabeth Roddar,Nadia Nadim,Simone Boye-Sørensen,Lotta Ökvist,Asato Miyagawa,Vilde Hasund,...,Bella Andersson,Bea Sprung,Suzu Amano,Smilla Holmberg,Lykke Ihrfelt,Vera Elin Kristina Blom,Doris Petz,Sally Nylén,Stella Maiquez,Fanny Peterson
None,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
None,0.0,1.000000,0.095502,0.408579,0.677857,0.380496,0.570724,0.156205,0.588926,0.291217,...,0.779116,0.343832,0.254828,0.758039,0.056343,0.290079,0.076391,-0.032464,-0.069641,0.029293
Jonna Ann-Charlotte Andersson,0.0,0.095502,1.000000,0.464201,0.514729,0.260662,0.173946,0.618630,0.485619,0.642613,...,0.572250,0.236080,0.217259,0.223725,0.124410,-0.000225,0.393898,-0.057073,0.032063,0.056320
Ellen Gibson,0.0,0.408579,0.464201,1.000000,0.758201,0.460788,0.160310,0.155915,0.557995,0.418820,...,0.611974,0.258117,0.471422,0.486706,-0.000823,0.228130,-0.090351,-0.110780,-0.041432,0.076455
Julia Elisabeth Roddar,0.0,0.677857,0.514729,0.758201,1.000000,0.530779,0.465579,0.312289,0.836707,0.552973,...,0.803509,0.641995,0.443350,0.812220,0.093008,0.262663,0.127135,-0.097248,0.169185,0.094748


In [ ]:
pearson_df.to_csv('pearson_matrix.csv', index=False)

In [17]:
'''
Do the players keep within positions?
'''

'\nDo the players keep within positions?\n'

In [ ]:
c, Q = fn.modularity(A, idx_to_player_id, df_stories, role_col='role')
print(Q)

-0.0013827293796566606


In [19]:
'''
The rest of the metrics
'''

'\nThe rest of the metrics\n'

In [ ]:
def as_1d(result):
    return np.asarray(result).ravel()

def safe_metric_call(name, func):
    result = func()
    return as_1d(result)

metrics = {
    'Eigenvector': lambda: fn.eigenvector_centrality_iter(A),
    'Katz': lambda: fn.katz_centrality_iter(A),
    'PageRank': lambda: fn.page_rank_iter(P),
    'Authorities': lambda: fn.authorities_hubs_iter(A)[0],
    'Hubs': lambda: fn.authorities_hubs_iter(A)[1],
    'Closeness': lambda: fn.closeness_centrality_simple(D),
    'Betweenness': lambda: fn.betweenness_centrality(A, normalized=True),
    'Local Clustering': lambda: fn.local_clustering(A, min_passes=40),
    'Reciprocity': lambda: fn.reciprocity_local(P),
    'Structural Balance local': lambda: fn.structural_balance_local(S),
    'Pearson': lambda: fn.pearson_matrix(P, mode='out').sum(axis=1),
    'Modularity': lambda: fn.modularity(A, idx_to_player_id, df_stories, role_col='role')[0]
    }

results = {'player_id': idx_to_player_id}
results['player_name'] = [id_to_name.get(pid, None) for pid in idx_to_player_id]

errors = {}
for name, func in metrics.items():
    results[name] = safe_metric_call(name, func)

metrics_df = pd.DataFrame(results)

metrics_df = metrics_df.sort_values('Eigenvector', ascending=False)

metrics_df.head(10), errors

(    player_id          player_name  Eigenvector       Katz   PageRank  \
 12     401725       Alice Carlsson     0.456429  18.310483  25.477569   
 23     566534  Emilie Marie Joramo     0.338375  13.716220  17.926789   
 20     535334       Julie Blakstad     0.327919  13.464271  21.140551   
 34     813153      Smilla Holmberg     0.299083  12.162527  14.693089   
 10     351320          Eva Nyström     0.292694  11.880138  12.038587   
 8      302840       Asato Miyagawa     0.237127   9.825676  11.832265   
 21     535335      Emilie Bragstad     0.213230   8.953120   9.732728   
 30     749906      Smilla Vallotto     0.208163   8.821897  13.032954   
 24     676040     Ellen Wangerheim     0.203406   8.674484  13.298755   
 14     418723    Stina Lennartsson     0.197744   8.369266  10.368674   
 
     Authorities      Hubs  Closeness  Betweenness  Local Clustering  \
 12     0.447440  0.545511   0.247102     0.568590          0.359477   
 23     0.330947  0.376388   0.243251   

In [ ]:
metrics_df.to_csv('Results.csv', index=False)